In [1]:
!pip install langchain langchain-community langchain-huggingface langchain-groq langchain-chroma chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.

In [2]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("groq")

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!pip install langchain-schema

ERROR: Could not find a version that satisfies the requirement langchain-schema (from versions: none)
ERROR: No matching distribution found for langchain-schema


In [5]:
import json
from langchain_core.documents import Document

json_path = "/content/drive/MyDrive/bcorp_extracted.json"

with open(json_path) as f:
    raw = json.load(f)

documents = [
    Document(
        page_content=json.dumps(company, indent=2),
        metadata={"source_file": filename}
    )
    for filename, company in raw.items()
]

print(f"Loaded {len(documents)} documents")

Loaded 7089 documents


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [7]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    persist_directory="./bcorp_chroma_db",
    collection_name="bcorp_companies"
)

In [9]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [10]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=1024
)

In [11]:
test_response = llm.invoke("Say hello in one sentence.")


In [12]:
print(test_response)

content="Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss." additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 41, 'total_tokens': 67, 'completion_time': 0.055253785, 'completion_tokens_details': None, 'prompt_time': 0.001459374, 'prompt_tokens_details': None, 'queue_time': 0.008748499, 'total_time': 0.056713159}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f8df8-a042-71a1-91f0-1c4ae7e59bc8-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 41, 'output_tokens': 26, 'total_tokens': 67}


In [13]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT_TEMPLATE = """\
You are an expert assistant on B Corp certified companies.
Use ONLY the context below to answer the user's question.
If the answer is not in the context, say "I don't have that information in the dataset."
Be concise, specific, and cite company names when relevant.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

prompt = ChatPromptTemplate.from_template(RAG_PROMPT_TEMPLATE)



In [14]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)



In [15]:
def ask(question: str) -> str:
    print(f"\n Question: {question}")
    print("-" * 60)
    answer = rag_chain.invoke(question)
    print(f" Answer: {answer}")
    print("-" * 60)
    return answer


In [16]:
ask("Which companies are headquartered in Canada?")


 Question: Which companies are headquartered in Canada?
------------------------------------------------------------
 Answer: All the companies mentioned are headquartered in Canada, specifically: 
- IKE Enterprises in British Columbia, 
- Total Fabrication Inc. in Quebec, 
- Épicerie LOCO in Quebec, 
- OnwardUP Sales and Marketing Ltd in Alberta, and 
- HealthPRO Canada in Ontario.
------------------------------------------------------------


'All the companies mentioned are headquartered in Canada, specifically: \n- IKE Enterprises in British Columbia, \n- Total Fabrication Inc. in Quebec, \n- Épicerie LOCO in Quebec, \n- OnwardUP Sales and Marketing Ltd in Alberta, and \n- HealthPRO Canada in Ontario.'

In [17]:
ask("Tell me about companies in the financial services industry.")


 Question: Tell me about companies in the financial services industry.
------------------------------------------------------------
 Answer: In the financial services industry, there are companies like Polestar, a mid-market corporate finance boutique, and Lombard Odier, a private bank focused on investment advising. Additionally, Finli Inc operates in financial transaction processing, and Financial Insights Wealth Management provides investment advising services. FLUX.FINANCIERA is also a financial intermediary. These companies, such as Polestar and Lombard Odier, have B Corp certifications, with scores like 80.2 and 111.2, respectively.
------------------------------------------------------------


'In the financial services industry, there are companies like Polestar, a mid-market corporate finance boutique, and Lombard Odier, a private bank focused on investment advising. Additionally, Finli Inc operates in financial transaction processing, and Financial Insights Wealth Management provides investment advising services. FLUX.FINANCIERA is also a financial intermediary. These companies, such as Polestar and Lombard Odier, have B Corp certifications, with scores like 80.2 and 111.2, respectively.'

In [ ]:
import shutil

# Copy the ChromaDB folder from Colab to Google Drive
shutil.copytree(
    "./bcorp_chroma_db",
    "/content/drive/MyDrive/bcorp_chroma_db"
)

print("ChromaDB saved to Google Drive!")

In [21]:
!pip install pyngrok

In [23]:

from google.colab import drive
drive.mount('/content/drive')

import json
from langchain_core.documents import Document

json_path = "/content/drive/MyDrive/bcorp_extracted.json"

with open(json_path) as f:
    raw = json.load(f)

documents = [
    Document(
        page_content=json.dumps(company, indent=2),
        metadata={
            "source_file": filename,
            "company_name": company.get("company_name", "Unknown"),
            "headquarters": company.get("headquarters", "Unknown"),
            "certified_since": company.get("certified_since", "Unknown"),
            "industry": company.get("industry", "Unknown"),
            "sector": company.get("sector", "Unknown"),
            "operates_in": ", ".join(company.get("operates_in", [])),
            "website": company.get("website", "Unknown"),
            "b_impact_score": float(company.get("b_impact_score") or 0.0),
            "governance_score": float(company.get("governance_score") or 0.0),
        }
    )
    for filename, company in raw.items()
]

print(f"Loaded {len(documents)} documents")

# Cell 3: build the vector store
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    persist_directory="./bcorp_chroma_db",
    collection_name="bcorp_companies"
)

print(f"ChromaDB built with {vectorstore._collection.count()} vectors!")

# build the RAG chain (with chat memory)
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from google.colab import userdata

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=1024,
    api_key=userdata.get('groq')
)

def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)


condense_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Given the conversation history and a follow-up question, rewrite the follow-up "
     "into a standalone question that includes any necessary context from the history. "
     "If the follow-up is already standalone, return it unchanged. "
     "Only output the rewritten question, nothing else."),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}"),
])

condense_chain = condense_prompt | llm | StrOutputParser()

def get_standalone_question(input_dict):
    if not input_dict["chat_history"]:
        return input_dict["question"]
    return condense_chain.invoke(input_dict)

def retrieve_and_format(input_dict):
    standalone_question = get_standalone_question(input_dict)
    docs = retriever.invoke(standalone_question)
    return format_docs(docs)

# answer using retrieved context + chat history
answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert assistant on B Corp certified companies. "
     "Use ONLY the context below to answer the question. "
     "If the answer is not in the context, say I don't have that information.\n\n"
     "CONTEXT:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}"),
])

rag_chain = (
    RunnablePassthrough.assign(context=RunnableLambda(retrieve_and_format))
    | answer_prompt | llm | StrOutputParser()
)

print("RAG chain with memory ready!")


#  FastAPI server wired to rag_chain
import threading

import nest_asyncio
import uvicorn
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok
from langchain_core.messages import HumanMessage, AIMessage

app = FastAPI()


MAX_HISTORY_MESSAGES = 10

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


class ChatRequest(BaseModel):
    messages: list


def to_lc_messages(messages):
    lc_messages = []
    for m in messages:
        role = m.get("role")
        content = m.get("content", "")
        if role == "user":
            lc_messages.append(HumanMessage(content=content))
        elif role == "assistant":
            lc_messages.append(AIMessage(content=content))
    return lc_messages


@app.post("/chat")
def chat(req: ChatRequest):

    if not req.messages or req.messages[-1].get("role") != "user":
        return {"reply": "I didn't receive a question."}

    question = req.messages[-1].get("content", "")
    history = req.messages[:-1][-MAX_HISTORY_MESSAGES:]
    chat_history = to_lc_messages(history)

    try:
        answer = rag_chain.invoke({"question": question, "chat_history": chat_history})
    except Exception as e:
        return {"reply": f"Something went wrong on the backend: {e}"}

    return {"reply": answer}


@app.get("/health")
def health():
    return {"status": "ok"}



ngrok.set_auth_token("")


import time


def _run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")


server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()
time.sleep(2)

public_url = ngrok.connect(8000)
print("=" * 60)
print("PASTE THIS INTO API_URL IN YOUR HTML:")
print(public_url.public_url)
print("=" * 60)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 7134 documents


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

ChromaDB built with 28486 vectors!
RAG chain with memory ready!


INFO:     Started server process [4203]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


PASTE THIS INTO API_URL IN YOUR HTML:
https://nonfelicitous-reasonedly-cadence.ngrok-free.dev
